In [1]:
import re
import numpy as np
import pandas as pd
import gensim.downloader as api
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss

In [2]:
print("Начало загрузки модели word2vec...")
w2v_model = api.load("glove-wiki-gigaword-300")
print("Модель успешно загружена!")

Начало загрузки модели word2vec...
Модель успешно загружена!


In [3]:
def handle_text(raw_text):
    text = str(raw_text).lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:
def get_sentence_vector(text, w2v, idf_map, default_idf):
    words = str(text).split()

    weighted_embeddings = []
    weights = []

    for word in words:
        if word in w2v.key_to_index:
            weight = idf_map.get(word, default_idf)
            weights.append(weight)
            weighted_embeddings.append(w2v[word] * weight)

    if len(weighted_embeddings) == 0:
        return np.zeros(w2v.vector_size)

    return np.sum(weighted_embeddings, axis=0) / np.sum(weights)

In [5]:
def generate_features(df, w2v, idf_dict, default_idf):
    all_features = []

    for idx, row in df.iterrows():
        text1 = str(row['q1_clean'])
        text2 = str(row['q2_clean'])
        words1 = text1.split()
        words2 = text2.split()

        v1 = get_sentence_vector(text1, w2v, idf_dict, default_idf)
        v2 = get_sentence_vector(text2, w2v, idf_dict, default_idf)

        sim = cosine_similarity(v1.reshape(1, -1), v2.reshape(1, -1))[0][0]

        len_q1 = len(text1)
        len_q2 = len(text2)
        len_diff = abs(len_q1 - len_q2)

        euclidean_dist = np.linalg.norm(v1 - v2)
        manhattan_dist = np.sum(np.abs(v1 - v2))

        q_words = ['what', 'how', 'why', 'who', 'where', 'when', 'which']
        first_word1 = words1[0] if len(words1) > 0 else ""
        first_word2 = words2[0] if len(words2) > 0 else ""
        q_type_match = 1.0 if (first_word1 == first_word2) and (first_word1 in q_words) else 0.0

        nums1 = set(re.findall(r'\d+', text1))
        nums2 = set(re.findall(r'\d+', text2))
        exact_number_match = 1.0 if nums1 == nums2 else 0.0

        row_features = [
            sim,
            len_diff,
            euclidean_dist,
            manhattan_dist,
            q_type_match,
            exact_number_match
        ]
        all_features.append(row_features)

    return np.array(all_features)

In [6]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

train['question1'] = train['question1'].fillna("")
train['question2'] = train['question2'].fillna("")

train['q1_clean'] = train['question1'].apply(handle_text)
train['q2_clean'] = train['question2'].apply(handle_text)

train_df = train.drop(columns=['is_duplicate'])
y = train['is_duplicate']

In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_preds = np.zeros(len(train_df))
fold_loss = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y)):
    fold_train = train_df.iloc[train_idx]
    fold_val = train_df.iloc[val_idx]

    train_corpus = fold_train['q1_clean'].tolist() + fold_train['q2_clean'].tolist()

    tfidf = TfidfVectorizer(lowercase=True)
    tfidf.fit(train_corpus)

    idf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
    default_idf = sum(tfidf.idf_) / len(tfidf.idf_)

    X_train = generate_features(fold_train, w2v_model, idf_dict, default_idf)
    X_val = generate_features(fold_val, w2v_model, idf_dict, default_idf)

    y_train = y[train_idx]
    y_val = y[val_idx]

    forest = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    forest.fit(X_train, y_train)

    preds = forest.predict_proba(X_val)[:, 1]
    all_preds[val_idx] = preds

    loss = log_loss(y_val, preds)
    fold_loss.append(loss)

    print(f"Ошибка на фолде {fold}: {loss}")

print(f"Средняя ошибка по фолдам: {np.mean(fold_loss)}")

Ошибка на фолде 0: 0.5462228131580119
Ошибка на фолде 1: 0.5483725390682587
Ошибка на фолде 2: 0.5495870557935804
Ошибка на фолде 3: 0.5472333012574817
Ошибка на фолде 4: 0.5474220022475211
Средняя ошибка по фолдам: 0.5477675423049708
